# COMET seed 123 — A/B

Venv Python 3.12 riêng (comet yêu cầu numpy<2), GPU + internet.


In [ ]:
import subprocess, sys, os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.12', '/kaggle/working/venv'], check=True)
P = '/kaggle/working/venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', P,
                'torch', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', P,
                'numpy<2', 'pandas==2.2.3', 'pyarrow>=17',
                'unbabel-comet>=2.2', 'sacrebleu'], check=True)
print('venv ready')


In [ ]:
from pathlib import Path
Path('/kaggle/working/score.py').write_text('\nimport json, os\nos.environ["USE_TF"] = "0"\nos.environ["USE_FLAX"] = "0"\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom comet import download_model, load_from_checkpoint\nimport sacrebleu\n\nprint("torch", torch.__version__, "cuda", torch.cuda.is_available())\nif not torch.cuda.is_available():\n    raise SystemExit("no GPU")\n\nDA = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))\nQE = load_from_checkpoint(download_model("Unbabel/wmt20-comet-qe-da"))\nprint("models loaded")\n\nHYP = Path("/kaggle/input/tiki-seed123-hyps")\nOUT = Path("/kaggle/working/scores"); OUT.mkdir(parents=True, exist_ok=True)\nmetrics = {}\nfor suite in ("A", "B"):\n    out_dir = OUT / suite; out_dir.mkdir(exist_ok=True)\n    for path in sorted(HYP.glob(f"hyp_{suite}_*.parquet")):\n        df = pd.read_parquet(path).drop_duplicates("eval_id")\n        config = str(df["config"].iloc[0]); seed = int(df["seed"].iloc[0])\n        for dataset in ("1688", "tiki"):\n            part = df[df["dataset"] == dataset].copy()\n            part = part[part["hypothesis"].astype(str).str.strip().ne("")]\n            if dataset == "1688":\n                part = part[part["reference"].astype(str).str.strip().ne("")]\n                data = [{"src": str(s), "mt": str(m), "ref": str(r)}\n                        for s, m, r in zip(part.source, part.hypothesis, part.reference)]\n                out = DA.predict(data, batch_size=16, gpus=1)\n                bleu = sacrebleu.corpus_bleu(part.hypothesis.tolist(),\n                                             [part.reference.tolist()], tokenize="zh").score\n                chrf = sacrebleu.corpus_chrf(part.hypothesis.tolist(),\n                                             [part.reference.tolist()]).score\n                extra = {"bleu": round(float(bleu), 4), "chrf": round(float(chrf), 4)}\n            else:\n                data = [{"src": str(s), "mt": str(m)}\n                        for s, m in zip(part.source, part.hypothesis)]\n                out = QE.predict(data, batch_size=16, gpus=1)\n                extra = {}\n            part["score"] = np.asarray(out.scores, dtype=float)\n            part["config"] = config; part["seed"] = seed; part["dataset"] = dataset\n            part[["eval_id", "pair_type", "score", "config", "seed", "dataset"]].to_parquet(\n                out_dir / f"scores_{config}_seed{seed}_{dataset}.parquet", index=False)\n            key = f"{suite}::{config}::{seed}::{dataset}"\n            metrics[key] = {"system_score": round(float(out.system_score), 4),\n                            "n": int(len(part)), **extra}\n            print(key, metrics[key], flush=True)\n            (OUT / "comet_scores.json").write_text(\n                json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\nprint(json.dumps(metrics, ensure_ascii=False, indent=2))\n', encoding='utf-8')
print('score.py written')


In [ ]:
import subprocess
r = subprocess.run(['/kaggle/working/venv/bin/python', '/kaggle/working/score.py'],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-4000:])
r.check_returncode()


In [ ]:
from pathlib import Path
OUT = Path('/kaggle/working/scores')
for p in sorted(OUT.rglob('*.parquet')): print(p, p.stat().st_size)
print((OUT / 'comet_scores.json').read_text()[:800])
